# Adversarial Edits (T = 1)

This notebook replaces the tokens with the largest pre-edit pivots, then decodes and re-tokenizes the text, recomputes post-edit pivots, and constructs the single token-level ground-truth label `GT`.

The output `post_edit_adversarial_T1.zip` contains edit budgets K in `{5, 10, 15, 20, 30, 40}`.


In [ ]:
%pip -q install transformers sentencepiece accelerate tqdm


In [ ]:
import json
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer


torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
def locate_zip(filename: str) -> Path:
    candidates = [Path('/content') / filename, Path('/mnt/data') / filename, Path.cwd() / filename]
    for path in candidates:
        if path.exists() and zipfile.is_zipfile(path):
            return path

    try:
        from google.colab import files
        uploaded = files.upload()
        for uploaded_name in uploaded:
            path = Path('/content') / uploaded_name
            if path.suffix.lower() == '.zip' and zipfile.is_zipfile(path):
                return path
    except Exception:
        pass

    raise FileNotFoundError(
        f'Could not find {filename}. Place it in /content, /mnt/data, or the current directory.'
    )


def load_dataset_zip(zip_path: Path, workdir_name: str):
    workdir_root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    workdir = workdir_root / workdir_name
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(workdir)

    dataset_paths = list(workdir.rglob('dataset.npz'))
    meta_paths = list(workdir.rglob('meta.json'))
    if len(dataset_paths) != 1:
        raise FileNotFoundError(f'Expected one dataset.npz in {zip_path}; found {len(dataset_paths)}.')
    if len(meta_paths) != 1:
        raise FileNotFoundError(f'Expected one meta.json in {zip_path}; found {len(meta_paths)}.')

    data = np.load(dataset_paths[0], allow_pickle=False)
    meta = json.loads(meta_paths[0].read_text(encoding='utf-8'))
    return data, meta, workdir


def save_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def save_dataset_zip(output_name: str, meta: dict, arrays: dict, extra_files: dict | None = None) -> Path:
    root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    folder = root / Path(output_name).stem
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

    save_json(folder / 'meta.json', meta)
    np.savez_compressed(folder / 'dataset.npz', **arrays)
    if extra_files:
        for name, payload in extra_files.items():
            path = folder / name
            if isinstance(payload, str):
                path.write_text(payload, encoding='utf-8')
            else:
                save_json(path, payload)

    zip_path = root / output_name
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=folder)
    print('Saved:', zip_path)

    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
    return zip_path


In [ ]:
PRE_EDIT_ZIP_NAME = 'pre_edit_T1.zip'
pre_edit_zip = locate_zip(PRE_EDIT_ZIP_NAME)
data, meta, _ = load_dataset_zip(pre_edit_zip, 'pre_edit_T1_extracted')

required_keys = {'tokens_gen', 'S1', 'Ys', 'prompts_tokens'}
missing = required_keys.difference(data.files)
if missing:
    raise KeyError(f'Missing pre-edit arrays: {sorted(missing)}')
if float(meta.get('temp', 1.0)) != 1.0:
    raise ValueError('This notebook is configured for the T = 1 dataset.')

tokens_gen = data['tokens_gen'].astype(np.int32)
S1 = data['S1'].astype(np.int8)
Ys = data['Ys'].astype(np.float32)
prompts_tokens = data['prompts_tokens'].astype(np.int32)
NUM_DOCUMENTS, GENERATION_LENGTH = tokens_gen.shape

MODEL_NAME = meta.get('model', 'facebook/opt-1.3b')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

TOKENIZER_VOCAB_SIZE = int(tokenizer.vocab_size)
TRUNCATE_VOCAB = int(meta.get('truncate_vocab', 8))
EFFECTIVE_VOCAB_SIZE = TOKENIZER_VOCAB_SIZE - TRUNCATE_VOCAB
FULL_VOCAB_SIZE = int(meta.get('vocab_size_full', max(TOKENIZER_VOCAB_SIZE, int(tokens_gen.max()) + 1)))
CONTEXT_WIDTH = int(meta.get('c_window', 5))
WATERMARK_KEY = int(meta.get('key', 15485863))
SEEDING_SCHEME = meta.get('seeding_scheme', 'noncomm_prf')
PAD_ID = 0

tokens_clipped = np.minimum(tokens_gen, EFFECTIVE_VOCAB_SIZE - 1).astype(np.int32)
print('Input:', pre_edit_zip)
print('Documents:', NUM_DOCUMENTS, '| Generation length:', GENERATION_LENGTH)


In [ ]:
def decode_tokens(token_ids: np.ndarray) -> str:
    return tokenizer.decode([int(x) for x in token_ids.tolist()], skip_special_tokens=True)


def encode_and_pad(text: str, target_length: int, pad_id: int = 0) -> tuple[np.ndarray, int]:
    encoded = tokenizer(text, truncation=True, max_length=2048)
    token_ids = np.asarray(encoded['input_ids'], dtype=np.int32)
    length = int(token_ids.size)
    if length <= target_length:
        padded = np.pad(token_ids, (target_length - length, 0), constant_values=pad_id)
        return padded.astype(np.int32), length
    return token_ids[:target_length].astype(np.int32), target_length


In [ ]:
import difflib

_rng_table = torch.Generator(device='cpu')
_rng_table.manual_seed(2971215073)
_TABLE_SIZE = 1_000_003
_FIXED_TABLE = torch.randperm(_TABLE_SIZE, device='cpu', generator=_rng_table)


def _hashint(integer_tensor: torch.LongTensor) -> torch.LongTensor:
    return _FIXED_TABLE[integer_tensor.cpu() % _TABLE_SIZE] + 1


def _noncomm_prf(input_ids: torch.LongTensor, salt_key: int) -> int:
    key_value = torch.as_tensor(int(salt_key), dtype=torch.long)
    for entry in input_ids:
        key_value *= _hashint(key_value * entry)
        key_value %= 2**32 - 1
    return int(key_value.item())


def _seed_rng(generator: torch.Generator, tokens_1xL: torch.LongTensor,
              seeding_scheme: str, hash_key: int, context_width: int) -> None:
    if seeding_scheme != 'noncomm_prf':
        raise ValueError("This notebook supports seeding_scheme='noncomm_prf'.")
    if tokens_1xL.shape[-1] < context_width:
        raise ValueError('The prefix is shorter than the watermark context width.')
    seed = _noncomm_prf(tokens_1xL[0, -context_width:], salt_key=hash_key)
    generator.manual_seed(seed)


def recompute_gumbel_pivots(text_ids: np.ndarray, prompt_ids: np.ndarray,
                            vocab_size: int, key: int, context_width: int,
                            seeding_scheme: str) -> np.ndarray:
    generator = torch.Generator(device='cpu')
    prompt_tail = torch.as_tensor(prompt_ids[-context_width:], dtype=torch.long)
    text = torch.as_tensor(np.asarray(text_ids, dtype=np.int64), dtype=torch.long)
    full_sequence = torch.cat([prompt_tail, text], dim=0)

    pivots = np.empty(text.shape[0], dtype=np.float32)
    for position in range(text.shape[0]):
        prefix = full_sequence[:context_width + position].unsqueeze(0)
        _seed_rng(generator, prefix, seeding_scheme, key, context_width)
        xi = torch.rand((vocab_size,), generator=generator)
        token_id = int(full_sequence[context_width + position].item())
        pivots[position] = float(xi[token_id].item()) if 0 <= token_id < vocab_size else np.nan
    return pivots


def ground_truth_from_alignment(post_ids_padded: np.ndarray, pre_ids: np.ndarray,
                                pre_labels: np.ndarray, context_width: int,
                                pad_id: int = 0) -> np.ndarray:
    post_ids_padded = np.asarray(post_ids_padded, dtype=np.int32).reshape(-1)
    pre_ids = np.asarray(pre_ids, dtype=np.int32).reshape(-1)
    pre_labels = np.asarray(pre_labels, dtype=np.int8).reshape(-1)

    first_valid = 0
    while first_valid < post_ids_padded.size and post_ids_padded[first_valid] == pad_id:
        first_valid += 1

    post_sequence = post_ids_padded[first_valid:].tolist()
    pre_sequence = pre_ids.tolist()
    matcher = difflib.SequenceMatcher(a=pre_sequence, b=post_sequence, autojunk=False)

    post_to_pre = np.full(len(post_sequence), -1, dtype=np.int32)
    for pre_start, post_start, length in matcher.get_matching_blocks():
        if length > 0:
            post_to_pre[post_start:post_start + length] = np.arange(
                pre_start, pre_start + length, dtype=np.int32
            )

    labels = np.zeros(len(post_sequence), dtype=np.int8)
    for post_index, pre_index in enumerate(post_to_pre):
        if pre_index < 0:
            continue
        valid = True
        for lag in range(context_width + 1):
            post_lag = post_index - lag
            pre_lag = pre_index - lag
            if post_lag < 0 or pre_lag < 0:
                valid = False
                break
            if post_to_pre[post_lag] != pre_lag:
                valid = False
                break
            if pre_lag >= pre_labels.size or pre_labels[pre_lag] == 0:
                valid = False
                break
        if valid:
            labels[post_index] = 1

    output = np.zeros(post_ids_padded.size, dtype=np.int8)
    output[first_valid:] = labels
    return output


## Apply the edits and save the post-edit dataset


In [ ]:
OUTPUT_NAME = 'post_edit_adversarial_T1.zip'
EDIT_BUDGETS = [5, 10, 15, 20, 30, 40]
TARGET_LENGTH = GENERATION_LENGTH

arrays = {
    'tokens_gen': tokens_gen,
    'S1': S1,
    'Ys': Ys,
    'prompts_tokens': prompts_tokens,
}

for budget in EDIT_BUDGETS:
    started = time.perf_counter()
    rng = np.random.default_rng(12345 + int(budget))
    edited_tokens = tokens_clipped.copy()
    for document_index in range(NUM_DOCUMENTS):
        effective_budget = min(int(budget), Ys.shape[1])
        positions = np.argpartition(Ys[document_index], -effective_budget)[-effective_budget:]
        edited_tokens[document_index, positions] = rng.integers(
            0, EFFECTIVE_VOCAB_SIZE, size=len(positions), dtype=np.int32
        )

    post_tokens = np.zeros((NUM_DOCUMENTS, TARGET_LENGTH), dtype=np.int32)
    GT = np.zeros((NUM_DOCUMENTS, TARGET_LENGTH), dtype=np.int8)
    post_pivots = np.zeros((NUM_DOCUMENTS, TARGET_LENGTH), dtype=np.float32)
    valid_lengths = np.zeros(NUM_DOCUMENTS, dtype=np.int32)

    for document_index in tqdm(range(NUM_DOCUMENTS), desc=f'K={budget}'):
        edited_text = decode_tokens(edited_tokens[document_index])
        token_ids, valid_length = encode_and_pad(edited_text, TARGET_LENGTH, PAD_ID)
        label = ground_truth_from_alignment(
            token_ids,
            tokens_clipped[document_index],
            S1[document_index],
            CONTEXT_WIDTH,
            PAD_ID,
        )
        pivots = recompute_gumbel_pivots(
            token_ids,
            prompts_tokens[document_index],
            FULL_VOCAB_SIZE,
            WATERMARK_KEY,
            CONTEXT_WIDTH,
            SEEDING_SCHEME,
        )
        post_tokens[document_index] = token_ids
        GT[document_index] = label
        post_pivots[document_index] = pivots
        valid_lengths[document_index] = valid_length

    arrays[f'tokens_post_adv_repo_k{budget}'] = post_tokens
    arrays[f'GT_adv_repo_k{budget}'] = GT
    arrays[f'Ys_post_adv_repo_k{budget}'] = post_pivots
    arrays[f'valid_len_adv_repo_k{budget}'] = valid_lengths
    arrays[f'frac_GT_adv_repo_k{budget}'] = GT.mean(axis=1).astype(np.float32)
    print(f'K={budget} construction time: {time.perf_counter() - started:.1f} seconds')

output_meta = dict(meta)
output_meta.update({
    'tag': 'post_edit_adversarial_T1',
    'edit_method': 'adversarial_edits',
    'top_k_list': EDIT_BUDGETS,
    'truncate_vocab': TRUNCATE_VOCAB,
    'eff_vocab_size': EFFECTIVE_VOCAB_SIZE,
    'target_len': TARGET_LENGTH,
    'GT_definition': 'Token-alignment ground truth on the final tokenization.',
    'repo_recompute_pivots': True,
})
save_dataset_zip(OUTPUT_NAME, output_meta, arrays)
